In [4]:
!pip install spacy
!pip install scikit-learn
!pip install PyPDF2
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
#import libraries
import spacy
import PyPDF2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
nlp = spacy.load("en_core_web_sm")

In [9]:
#Resume Pdf Reader
def extract_text_from_pdf(file):
    text = ""
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

    return text

In [8]:
#Text Preprocessing
def preprocess_text(text):
    doc = nlp(text.lower())
    tokens = []

    for token in doc:
        if not token.is_stop and token.is_alpha:
            tokens.append(token.lemma_)

    return " ".join(tokens)

In [7]:
#ATS Similarity Score
def calculate_similarity(resume, job_description):

    documents = [resume, job_description]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

    return similarity[0][0] * 100

In [11]:
#keyword extraction
def extract_keywords(text):

    doc = nlp(text)
    keywords = []

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "ADJ"]:
            keywords.append(token.text.lower())

    return list(set(keywords))

In [10]:
#Skill Gap Detection
def skill_gap(resume_keywords, job_keywords):

    missing = []

    for word in job_keywords:
        if word not in resume_keywords:
            missing.append(word)

    return missing[:10]

In [12]:
resume_text = """
Python machine learning developer with experience in pandas,
scikit-learn, docker, and data analysis.
"""

job_description = """
Looking for a machine learning engineer with Python,
TensorFlow, pandas, deep learning and data analysis experience.
"""

resume_clean = preprocess_text(resume_text)
job_clean = preprocess_text(job_description)

score = calculate_similarity(resume_clean, job_clean)

resume_keywords = extract_keywords(resume_text)
job_keywords = extract_keywords(job_description)

missing_skills = skill_gap(resume_keywords, job_keywords)

print("ATS Score:", round(score,2))
print("Missing Skills:", missing_skills)

ATS Score: 40.3
Missing Skills: ['deep', 'engineer', 'tensorflow', 'learning']
